In [ ]:
# GEE_Time_Series_Analysis.ipynb

# This notebook analyzes land use changes at Bitcoin mining locations
# using AlphaEarth satellite embeddings.

# Task 1: Consolidate Successful Code
# This first section contains the proven code from GEE-test.ipynb
# for setting up the environment, loading the mining locations,
# and creating a balanced set of training points.

import ee
import geemap

# Authenticate and initialize Earth Engine
# This may require you to sign in to your Google account.
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project="ee-ktwu01") # Replace with your GEE project if needed

print("Earth Engine initialized successfully!")


In [ ]:
# 1. Configuration and Data Loading

# --- Constants ---
# The GEE asset path for the Bitcoin mining locations FeatureCollection.
# This asset was created by uploading the CSV from: https://www.nature.com/articles/s41598-022-14987-0
MINING_LOCATIONS_ASSET = 'projects/ee-ktwu01/assets/bitcoin-mining'

# The AlphaEarth image collection for satellite embeddings.
ALPHAEARTH_COLLECTION = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'

# --- Load AlphaEarth Collection ---
alphaEarth = ee.ImageCollection(ALPHAEARTH_COLLECTION)
print(f"Loaded AlphaEarth collection: {ALPHAEARTH_COLLECTION}")


# --- Load and Format Mining Locations ---
def format_mining_location(feature):
    """Casts properties from the asset to the correct type and structure."""
    lat = ee.Number(feature.get('Latitude'))
    lon = ee.Number(feature.get('Longitude'))
    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'label': 1, # 1 indicates a positive (mining) location
            'year': 2018, # The year the data was published
            'location': feature.get('CRCode'),
            'country_name': feature.get('CRName')
        }
    )

# Load the raw feature collection from the asset
miningLocations = ee.FeatureCollection(MINING_LOCATIONS_ASSET)

# Map the formatting function over the collection
miningLocations = miningLocations.map(format_mining_location)

# Get the number of positive samples
numPositive = miningLocations.size()
print('Number of mining locations loaded:', numPositive.getInfo())


In [ ]:
# 2. Generate Negative Samples

# To create a balanced dataset for any potential classification tasks,
# we generate an equal number of "negative" samples (random points
# that are not known mining locations).

# Get the geographic bounds of the mining locations to constrain the random sampling
miningBounds = miningLocations.geometry().bounds()

# Generate random points within the same geographic region
negativeLocations = ee.FeatureCollection.randomPoints(
    region=miningBounds,
    points=numPositive,  # Match the number of positive samples
    seed=42              # Use a seed for reproducibility
)

# Add a 'label' property to the negative samples
def add_negative_label(feature):
    return feature.set({
        'label': 0, # 0 indicates a negative (non-mining) location
        'year': 2018,
        'location': 'negative_sample'
    })

negativeLocations = negativeLocations.map(add_negative_label)

print('Number of negative samples generated:', negativeLocations.size().getInfo())

# --- Combine Positive and Negative Samples ---
# Merge the two feature collections into a single collection.
trainingPoints = miningLocations.merge(negativeLocations)
print('Total training points (positive + negative):', trainingPoints.size().getInfo())
